# Curved interfaces as operators

A field in a homogeneous medium *is* its angular spectrum.  A flat interface acts
diagonally on that spectrum — one Fresnel dyadic per direction.  A curved
interface does not, and `vectorwave` computes the operator that replaces it:
locally refract the field by the tangent plane, then radiate the surface currents
back into an outgoing spectrum by a surface transform.

Because every interface maps the space of spectra to itself, interfaces
**compose**.  A multi-element system is an ordered product of operators with free
propagation between them.  This notebook walks from a single surface up to a
two-element lens, and out to a genuinely non-axisymmetric freeform.

The sign convention: surfaces are graphs $z = \mathrm{sag}(\rho)$ with a
**positive radius placing the centre of curvature at $+z$**, so a beam going
toward $+z$ is converged by a *negative* radius.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")            # keep the rendered output tidy

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import PowerNorm

import vectorwave as vw

plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.titlesize": 10,
                     "figure.facecolor": "white"})
print("vectorwave", vw.__version__, "| packaged systems:", vw.available())

## 1 · The surface zoo

Every `Surface` knows its sag, slope and normal, so the propagators never care
which shape they were handed.  A conic constant $\kappa$ selects the type:
$\kappa=0$ sphere, $-1$ paraboloid, $<-1$ hyperboloid, $-1<\kappa<0$ ellipsoid.

In [ ]:
R = -8.0
shapes = {
    "sphere ($\\kappa$=0)": vw.Conic(R, 0.0),
    "paraboloid ($\\kappa$=-1)": vw.Conic(R, -1.0),
    "hyperboloid ($\\kappa$=-2.25)": vw.Conic(R, -2.25),
    "even asphere": vw.EvenAsphere(R, 0.0, coefficients=(2e-4, -3e-6)),
}
fig, ax = plt.subplots(figsize=(6.2, 3.6))
for name, s in shapes.items():
    rho, sag = s.profile(n=200, r_max=6.0)
    ax.plot(rho, sag, lw=1.8, label=name)
ax.set_xlabel(r"$\rho$ / $\lambda$"); ax.set_ylabel("sag / $\\lambda$")
ax.set_title(f"Surface profiles (vertex radius R = {R})")
ax.legend(fontsize=8); ax.grid(alpha=0.25); ax.invert_yaxis()
fig.tight_layout()

## 2 · The stigmatic focus, against full-wave FDTD

The Cartesian oval that focuses a collimated beam with **no spherical aberration
at any order** is the conic with $\kappa = -(n_1/n_2)^2$
(`vw.stigmatic_conic_constant`).  Refract a plane wave in glass ($n_1=1.5$)
through such a hyperboloid into air ($n_2=1$) and scan for the focus.

The paraxial prediction is $z = n_2 R_v /(n_1-n_2) = 16\,\lambda$.  The actual
focus sits **short** of that, near $14.5\,\lambda$, because the Fresnel number is
finite — and an independent full-wave FDTD (MEEP) run put it at the same place.
This is a pinned regression test of the package.

In [ ]:
n1, n2, Rv = 1.5, 1.0, 8.0
surf = vw.Conic(radius=-Rv, conic=vw.stigmatic_conic_constant(n1, n2))
print(f"stigmatic conic constant kappa = {surf.conic:.3f}")

grid = vw.Grid.from_spacing(0.25, 200)
spec = vw.surface_spectrum(surf, grid, n1=n1, n2=n2, wavelength=1.0,
                           aperture=0.62 * 2 * Rv, m_max=2, n_rho=700,
                           n_phi=32, n_kr=256)
z_paraxial = n2 * Rv / (n1 - n2)
zs = np.linspace(0.5 * z_paraxial, 1.4 * z_paraxial, 90)
scan = spec.focus_scan(zs)
zf = zs[np.argmax(scan)]

fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.5))
ax[0].plot(zs, scan / scan.max(), lw=1.8)
ax[0].axvline(zf, color="C1", lw=1.2, label=f"focus {zf:.1f} $\\lambda$")
ax[0].axvline(z_paraxial, color="k", ls="--", lw=1.1, label=f"paraxial {z_paraxial:.0f} $\\lambda$")
ax[0].set_xlabel("z / $\\lambda$"); ax[0].set_ylabel("on-axis intensity")
ax[0].set_title("focal shift from a finite Fresnel number"); ax[0].legend(fontsize=8)

xr = np.arange(-2.5, 2.5, 1 / 40)
f = spec.field_on(xr, xr, zf)
ax[1].imshow(f.intensity, extent=[xr[0], xr[-1], xr[0], xr[-1]], origin="lower",
             cmap="inferno", norm=PowerNorm(0.5))
ax[1].set_title(f"focal spot at z = {zf:.1f} $\\lambda$")
ax[1].set_xlabel("x / $\\lambda$"); ax[1].set_ylabel("y / $\\lambda$")
ax[1].set_xlim(-1.5, 1.5); ax[1].set_ylim(-1.5, 1.5)
fig.tight_layout()
print(f"transversality residual |k.A| = {spec.transversality_residual():.1e}  (physical field)")

## 3 · The interface as a composable operator

`surface_spectrum` above takes a *named* source (a plane wave).  `InterfaceOperator`
does the same physics but consumes and produces an `AngularSpectrum`, so it
**composes**.  Driven by a plane-wave spectrum it must reproduce `surface_spectrum`
exactly — the two paths agree on the focus and the focal field.

In [ ]:
pw = vw.plane_wave_spectrum(grid, wavelength=1.0, n=1.5, polarization="x")
op = vw.InterfaceOperator(surf, n1=1.5, n2=1.0, aperture=0.62 * 2 * Rv, n_rho=600)
out = op(pw)

zf_src = spec.best_focus(zs)
zf_op = out.best_focus(zs)
print(f"focus from surface_spectrum : {zf_src:.2f} lambda")
print(f"focus from InterfaceOperator: {zf_op:.2f} lambda   (agree to < 0.5 lambda)")

a = spec.field_on(xr, xr, zf_src).intensity
c = out.field_on(xr, xr, zf_op).intensity
corr = np.sum(a * c) / np.sqrt(np.sum(a * a) * np.sum(c * c))
print(f"focal-field correlation between the two paths: {corr:.4f}")

### Free space shifts the focus, exactly

`FreeSpace(d)` is the diagonal propagator.  Applying it before synthesis simply
slides the focus by $d$ — the operator algebra reads like the geometry.

In [ ]:
D = 4.0
zf0 = out.best_focus(zs)
zfD = vw.FreeSpace(D)(out).best_focus(zs - D)
print(f"focus of the bare spectrum        : {zf0:.2f} lambda")
print(f"focus after FreeSpace({D}) applied  : {zfD:.2f} lambda")
print(f"shift = {zf0 - zfD:.2f} lambda   (expected {D})")

## 4 · A system is a product of operators

Two refractions make a lens.  `System([front, FreeSpace(t), back])` is a word in
the operator algebra: a plane wave in air enters a glass cap, propagates through
the body, and exits the flat back face to a focus in air.

In [ ]:
n_g = 1.5
front = vw.InterfaceOperator(
    vw.Conic(radius=+6.0, conic=vw.stigmatic_conic_constant(1.0, n_g)),
    n1=1.0, n2=n_g, aperture=5.5)
back = vw.InterfaceOperator(vw.Plane(), n1=n_g, n2=1.0, aperture=5.5)
system = vw.System([front, vw.FreeSpace(8.0), back])
print("system:", system, f"({len(system)} operators)")

out = system(vw.plane_wave_spectrum(grid, wavelength=1.0, n=1.0))
zsys = np.linspace(2, 40, 140)
scan = out.focus_scan(zsys)
zf = zsys[np.argmax(scan)]

fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.4))
ax[0].plot(zsys, scan / scan.max(), lw=1.8)
ax[0].axvline(zf, color="C1", lw=1.2, label=f"focus {zf:.1f} $\\lambda$")
ax[0].set_xlabel("z / $\\lambda$"); ax[0].set_ylabel("on-axis intensity")
ax[0].set_title("two-surface lens: an interior focus"); ax[0].legend(fontsize=8)
xr2 = np.arange(-2.5, 2.5, 1 / 36)
f = out.field_on(xr2, xr2, zf)
ax[1].imshow(f.intensity, extent=[xr2[0], xr2[-1], xr2[0], xr2[-1]], origin="lower",
             cmap="inferno", norm=PowerNorm(0.5))
ax[1].set_title(f"focal spot at z = {zf:.1f} $\\lambda$")
ax[1].set_xlim(-1.5, 1.5); ax[1].set_ylim(-1.5, 1.5)
ax[1].set_xlabel("x / $\\lambda$"); ax[1].set_ylabel("y / $\\lambda$")
fig.tight_layout()
print(f"system output transversality: {out.transversality_residual():.1e}")

## 5 · Two return integrals, one answer

For a surface of revolution the return integral separates into azimuthal
harmonics (`method="polar"`, the fast Bessel kernel).  A general NUFFT of the
surface currents (`method="nufft"`) needs no symmetry at all.  On an axisymmetric
surface both are available and must agree — that overlap is what keeps the fast
path honest.

In [ ]:
a = vw.InterfaceOperator(surf, n1=1.5, n2=1.0, aperture=9.0, method="polar")(pw)
c = vw.InterfaceOperator(surf, n1=1.5, n2=1.0, aperture=9.0, method="nufft")(pw)
za, zc = a.best_focus(zs), c.best_focus(zs)
fa = a.field_on(xr, xr, za).intensity
fc = c.field_on(xr, xr, za).intensity
corr = np.sum(fa * fc) / np.sqrt(np.sum(fa * fa) * np.sum(fc * fc))
print(f"polar focus {za:.2f} lambda | nufft focus {zc:.2f} lambda")
print(f"focal-field correlation polar vs nufft: {corr:.4f}")

## 6 · A genuine freeform

Break the rotational symmetry.  `Freeform2D` takes any smooth `sag_fn(x, y)`; its
operator runs through the same NUFFT surface transform, so only the cost changes.
An **astigmatic** cap (different curvature in $x$ and $y$) focuses to two
separated line foci — the tell-tale of astigmatism, reproduced from first
principles.

In [ ]:
Rx, Ry = -14.0, -10.0                       # different meridional radii -> astigmatism
ff = vw.Freeform2D(sag_fn=lambda x, y: x ** 2 / (2 * Rx) + y ** 2 / (2 * Ry), radius=6.0)
outff = vw.InterfaceOperator(ff, n1=1.5, n2=1.0, aperture=6.0, n_free=200)(
    vw.plane_wave_spectrum(grid, wavelength=1.0, n=1.5))

xr3 = np.arange(-3, 3, 1 / 28)
zline = np.linspace(6, 26, 5)
fig, ax = plt.subplots(1, len(zline), figsize=(14, 3.0))
for a, z in zip(ax, zline):
    f = outff.field_on(xr3, xr3, z)
    a.imshow(f.intensity, extent=[xr3[0], xr3[-1], xr3[0], xr3[-1]], origin="lower",
             cmap="inferno", norm=PowerNorm(0.5))
    a.set_title(f"z = {z:.0f} $\\lambda$")
    a.set_xlim(-2, 2); a.set_ylim(-2, 2); a.set_xlabel("x / $\\lambda$")
ax[0].set_ylabel("y / $\\lambda$")
fig.suptitle("Astigmatic freeform: a horizontal line focus, then a vertical one", y=1.04)
fig.tight_layout()
print(f"freeform output transversality: {outff.transversality_residual():.1e}")

The beam squeezes to a **horizontal** line where the $y$-curvature focuses, opens
through a round-ish waist, and squeezes to a **vertical** line where the
$x$-curvature focuses.  Two line foci separated along $z$ — astigmatism — from a
surface described by nothing more than `x**2/(2Rx) + y**2/(2Ry)`.